# Replacing a physics process from the driver

Some of CAM's physics processes freeCAM owns as Python classes -- radiation,
the cloud macro/microphysics stage, dry adjustment, the convections, vertical
diffusion.  `driver.processes` is the table of them, bound to one run.
Looking a process up changes nothing.  Filling a slot on it does: the next
`advance` runs the process through its class, with whatever stands in the
slot answering its compute block -- a capture replayed, a network, or the
original Fortran called in place -- and the memory around the block read and
written from Python.  Empty every slot and the next `advance` is the original
Fortran path again, byte for byte.

Five cells: what this checkout has, the table before anything runs, a replay
in the radiation slot (bit-for-bit with the original), a trained network in
it (fast, and how far it drifts), and the two slots of the cloud stage.  The
512-rank gates behind each of these are in `docs/physics_kernel_decoupling.md`.

## 0. Does this checkout have what it needs?

The same check as `run_freecam.ipynb`: Derecho, the native image, a
configured case with one completed run.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / 'pyproject.toml').is_file():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))
from freecam import site

where = site.resolved(repo=REPO)
for name in ('account', 'queue', 'scratch', 'reference case', 'reference run'):
    print(f'{name:<16}{where[name]}')
print()

checks = site.preflight(repo=REPO)
for check in checks:
    print(f'  {check}')
absent = [check for check in checks if not check.ok]
assert not absent, (
    '\n\nnot ready.  What is missing, and what produces it:\n  '
    + '\n  '.join(f'{check.name}: {check.produced_by}' for check in absent)
    + '\n\nSee the README, section "Site configuration".')

## 1. The table, before anything runs

Constructing a `Driver` starts nothing.  The table lists the processes owned
as classes; asking for one binds the class to this run and nothing more.

In [ ]:
import freecam as fc

STEPS = int(os.environ.get('PYCAM_STEPS', 4))
SCRATCH = Path(where['scratch'])

driver = fc.Driver(case='PI-atm', nsteps=STEPS, restart_every='end')
print('processes owned as classes:')
for name in driver.processes:
    print('  ', name)

rad = driver.processes['radiation']          # the Radiation stage of this run
print()
print('radiation slot:', rad.process)        # None: the original Fortran computes it
print('who computes what:', driver.processes.describe())

## 2. A replay in the slot: the driver, proved

The block contract names what `radiation_tend` has in memory before its
arithmetic (`freecam.physics.radiation_process.BLOCK_INPUTS`) and the twelve
things it leaves behind.  A capture of a run holds every call's inputs and
outputs; replayed through the slot, the Python driver reads the inputs from
memory, writes the recorded outputs where the driver leaves them, and does the
bookkeeping around them itself -- and the state after fifty steps is bit-for-bit
the original's (7436003, 7438656).  That is the proof of the driver, with no
model in it.

Point `PYCAM_RAD_CAPTURE` at a capture directory (`--radiation-capture` in the
50-step job writes one, `radiation_tend.rank-NNNN.npz` per rank); by default
the newest one under the scratch root is taken.  The replay carries the ranks'
own files: the driver hands each rank its own.

In [ ]:
from freecam.physics.radiation_process import load_block_model


def newest(paths):
    ranked = sorted(paths, key=lambda p: p.stat().st_mtime)
    return ranked[-1] if ranked else None


capture = os.environ.get('PYCAM_RAD_CAPTURE') or newest((SCRATCH / 'pyCAM' / 'PI-cam').glob('rad-process-capture-*'))
assert capture, 'no radiation capture: run the 50-step job with PYCAM_RAD_CAPTURE=<dir> first'
print('capture:', capture)

rad.process = load_block_model(f'replay:{capture}', rank=0)   # each rank loads its own file when the stage reaches it
print('radiation slot:', rad.process)

driver.close()                      # release a model left over from a rerun
driver.initialize()                 # the 512-rank job: as long as the queue takes
print('run directory:', driver.run_dir)

before = driver.cam.state.T.stats(rank='global')
result = driver.run(steps=STEPS, progress=True)
after = driver.cam.state.T.stats(rank='global')
print(f"global mean T  {before['mean']:.4f} -> {after['mean']:.4f} K over {STEPS} steps")
print('who computed radiation:', driver.status['processes']['radiation'])

## 3. A network in the slot

Any Python callable over the block contract's inputs that returns the twelve
outputs can stand in the slot.  Here it is the MLP `train_rad_block.py` trains
on a capture, run as three NumPy matrix products (`rad_block_mlp.py`); on
512 ranks it answers a chunk in 1.5 ms and runs a month in 359.5 s against
400.2 for the original (7454279).  It is also, as trained, a poor model: the
block contract gives it no zenith angle, and by day 30 its net shortwave is
55 W/m² low.  The slot does not care; the drift is the model's to fix.

Point `FREECAM_RAD_BLOCK` at the weights' prefix (`<prefix>_weights.npz` and
`<prefix>_layout.json`); without it the cell says how to train them and skips.

In [ ]:
from freecam.physics.radiation_process import RadiationBlockModel

weights = os.environ.get('FREECAM_RAD_BLOCK')
if not weights:
    print('FREECAM_RAD_BLOCK is not set: train a block model first --')
    print('  examples/plugins/numba_kernels/train_rad_block.py <capture-dir> <out-prefix>')
else:
    sys.path.insert(0, str(REPO / 'examples' / 'plugins' / 'numba_kernels'))
    from rad_block_mlp import radiation_block          # loads the weights FREECAM_RAD_BLOCK names

    rad.process = RadiationBlockModel(radiation_block, label='block MLP')
    before = driver.cam.state.T.stats(rank='global')
    driver.advance(STEPS)                              # the slot changed: the stage re-attaches, the network answers
    after = driver.cam.state.T.stats(rank='global')
    print(f"global mean T  {before['mean']:.4f} -> {after['mean']:.4f} K over {STEPS} steps with the network")
    print('the slot, as the stage saw it:', rad.describe_process())

## 4. Back to the original

Empty the slot and the next `advance` detaches the stage: the workflow's
Fortran action is enabled again and radiation is the original, once, with no
runner and no Python in the step.

In [ ]:
rad.process = None
driver.advance(1)
print('who computes radiation now:', driver.status['processes'].get('radiation'))

## 5. Two slots on the cloud stage

The cloud macro/microphysics stage -- the one `mmacro_pcond` lives in -- is two
compute blocks with bookkeeping between them, so it has two slots: `process`
for the macrophysics driver's block and `micro_process` for the microphysics
driver's with its aerosol activation (`freecam.physics.cloud_block`).  Either
slot turns the stage into its Python driver.  Here the macrophysics block is
the original, called in place through the driver: the run stays bit-for-bit
(7451930) and the driver's own cost shows -- 0.37 s a rank per fifty steps,
about 4 ms a chunk a step.

Other answerers for either slot: `load_block_model('replay:DIR', block=...)`
for a capture made with `--cloud-block-capture`, `'verify:DIR'` to run the
original and compare what it wrote with the capture field by field,
`'census'` to run the original and name every buffer field it changed, or a
`BlockModel` over any callable.

In [ ]:
from freecam.physics.cloud_block import MACRO_BLOCK, MICRO_BLOCK, OriginalBlock

cloud = driver.processes['cloud_macro_microphysics']
print('macrophysics block:', len(MACRO_BLOCK.inputs), 'inputs,', len(MACRO_BLOCK.outputs), 'outputs')
print('microphysics block:', len(MICRO_BLOCK.inputs), 'inputs,', len(MICRO_BLOCK.outputs), 'outputs')

cloud.process = OriginalBlock(MACRO_BLOCK)   # the Python driver around the original macrophysics
cloud.micro_process = None                   # the microphysics driver original as well, in place
driver.advance(STEPS)
print('who computed the cloud stage:', driver.status['processes'].get('cloud_macro_microphysics'))
print('the slots, as the stage saw them:', cloud.describe_process())

cloud.process = None                         # both slots empty: the Fortran action is back

## 6. Release the allocation

`cpudev` allows one allocation per user; a model left held is one nobody else
can have.

In [ ]:
driver.close()
print('closed; the allocation is released')

## Where the same thing is on the command line

`--radiation-block-model replay:DIR | path.py:function`, `--cloud-block-model`
and `--cloud-micro-block-model` on `freecam` (with `--radiation-python` or
`--cloud-macro-micro-python`); the 50-step and month jobs take them as
`PYCAM_RAD_BLOCK_MODEL`, `PYCAM_CLOUD_BLOCK_MODEL`, `PYCAM_CLOUD_MICRO_BLOCK_MODEL`.
The other two ways a model reaches the radiation slot -- a Numba-compiled
plugin called by Fortran (`NativePlugin`) and a TorchScript file through
FTorch (`NativeModel`) -- go in the same `process` slot; what they cost, path
by path, is in `docs/physics_kernel_decoupling.md`.